# 21 Agreement, reliability and diagnostic tests on real data, with the packages' own examples

`table.agreement_between` (Cohen's, weighted and Fleiss' kappa, every intraclass correlation, Krippendorff, Lin),
`table.limits_of_agreement`, `table.reliability_of`, `table.compare_to_truth`, `sdata.compare_to_truth` and
`table.compare_accuracy`.

**Only simplydata runs in this notebook.** The reference values are computed by
`scripts/references/21_agreement_and_diagnostic_tests.py` and stored in
`references/21_agreement_and_diagnostic_tests.csv` with their source, kind and link: the coefficients written out
from their published formulas (Cohen 1960; Fleiss, Cohen & Everitt 1969; Fleiss 1971; McGraw & Wong 1996;
Krippendorff 2011, checked first on his own published example; Lin 1989; Bland & Altman 1986, 1999; Cronbach 1951;
Feldt 1965; DeLong et al. 1988; Simel et al. 1991), R irr, psych, pROC and `binom.test`, and the answer printed in
the text of R epiR's `epi.tests` help page (sensitivity 0.90, 0.88 to 0.92; specificity 0.76, 0.73 to 0.79;
positive likelihood ratio 3.75, 3.32 to 4.24; compared to half a unit of the printed digit).

Data: R irr's `diagnoses` (Fleiss 1971) and `anxiety`, the help-page examples of irr; pROC's `aSAH` (Turck et al.
2010); epiR's `epi.tests` example (Scott et al. 2008, Table 1); the serum and plasma creatinine of R mcr and three
items of Holzinger & Swineford (1939), stored in `data/`.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows, versions

NOTEBOOK = "21_agreement_and_diagnostic_tests"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def answer(word, *args, **options):
    return read_row(quiet(word, *args, **options))


def by_measure(word, *args, **options):
    """Each row of a one-row-per-measure answer, its numbers named "measure: column" (the AUC row as "AUC")."""
    values = {}
    for row in read_rows(quiet(word, *args, **options)):
        name = "AUC" if row["measure"] == "area under the ROC curve (AUC)" else row["measure"]
        values[name] = row["estimate"]
        for column in row:
            if column == "estimate" or (column.startswith("estimate ") and "CI" in column):
                values[f"{name}: {column}"] = row[column]
    return values

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Kappa: the psychiatric diagnoses of Fleiss (1971) and the anxiety ratings (R irr examples)

In [2]:
diagnoses = load("diagnoses")
raters = diagnoses.get_column_names()
diagnoses.agreement_between("rater1", "rater2")

"rater1" and "rater2" agree on 22 of 30 rows (73.3 %); by chance alone they would agree on about 23.6 %. Cohen's kappa = 0.651 (95 % CI 0.456 to 0.847): they agree 65 % of the way from chance to perfect.
If the two agreed only by chance, agreement this far above chance would turn up in fewer than 1 in 1000 studies like this one (p < 0.0001).
Method: Cohen's kappa (statsmodels cohens_kappa), as R irr::kappa2() and psych::cohen.kappa(). Kappa can be low even when the raw agreement is high, if one category is rare, so both are shown. Agreement is not correlation: two raters can rise and fall together and still disagree.


,test,what it compares,rows,raw agreement,agreement by chance,estimate,estimate 95 % CI low,estimate 95 % CI high,statistic name,statistic,p-value,method
1,Cohen's kappa,"""rater1"" and ""rater2""",30,0.733333,0.235556,0.651163,0.455788,0.846537,z,6.996471,2.624905e-12,"statsmodels cohens_kappa, as R irr::kappa2()"


In [3]:
ds = "diagnoses (R irr example; Fleiss 1971)"
checks.check_values(ds, "agreement_between (2 columns of categories: Cohen's kappa)",
                    answer(diagnoses.agreement_between, "rater1", "rater2"))
checks.check_values(ds, "agreement_between (3 or more columns of categories: Fleiss' kappa)",
                    answer(diagnoses.agreement_between, *raters))
anxiety_text = load("anx_text")
for near in ("linear", "quadratic"):
    found = answer(anxiety_text.agreement_between, "rater1", "rater2", near_misses=near, in_order=[f"rating {v}" for v in range(1, 7)])
    checks.check("anxiety (R irr example)", f'agreement_between (near_misses="{near}", in_order=)', "estimate", found["estimate"])

agrees  diagnoses (R irr example; Fleiss 1971) | agreement_between (2 columns of categories: Cohen's kappa) | estimate: simplydata 0.651162790698, Cohen (1960); Fleiss, Cohen & Everitt (1969), NumPy 0.651162790698; relative difference 0.00e+00 (allowed 1e-10)
      Source: Cohen (1960); Fleiss, Cohen & Everitt (1969), NumPy (run reference)
agrees  diagnoses (R irr example; Fleiss 1971) | agreement_between (2 columns of categories: Cohen's kappa) | estimate: simplydata 0.651162790698, R irr kappa2 (help page example) 0.651162790698; relative difference 0.00e+00 (allowed 1e-10)
      Source: R irr kappa2 (help page example) (run reference) - https://search.r-project.org/CRAN/refmans/irr/html/kappa2.html (version 0.85)
agrees  diagnoses (R irr example; Fleiss 1971) | agreement_between (2 columns of categories: Cohen's kappa) | estimate 95 % CI low: simplydata 0.455788374806, Cohen (1960); Fleiss, Cohen & Everitt (1969), NumPy 0.455788374806; relative difference 1.22e-16 (allowed 1e-10)
  

## The intraclass correlation in every form, Krippendorff's alpha and Lin's concordance correlation

In [4]:
anxiety = load("anxiety")
ds = "anxiety (R irr example)"
for who, checking, of in [("a sample", "agreement", "one rater"), ("a sample", "agreement", "average of raters"),
                          ("a sample", "consistency", "one rater"), ("a sample", "consistency", "average of raters"),
                          ("these only", "consistency", "one rater"), ("different per row", "agreement", "one rater"),
                          ("different per row", "agreement", "average of raters")]:
    checks.check_values(ds, f'agreement_between (raters="{who}", checking_for="{checking}", of="{of}")',
                        answer(anxiety.agreement_between, "rater1", "rater2", "rater3", raters=who, checking_for=checking, of=of))
for interval in (90, 99):
    checks.check_values(ds, f'agreement_between (checking_for="consistency", interval={interval})',
                        answer(anxiety.agreement_between, "rater1", "rater2", "rater3", checking_for="consistency", interval=interval))
for name, table, columns in [(ds, anxiety, ["rater1", "rater2", "rater3"]), ("diagnoses (R irr example; Fleiss 1971)", diagnoses, raters)]:
    found = answer(table.agreement_between, *columns, test_method="krippendorff")
    checks.check(name, 'agreement_between (test_method="krippendorff")', "estimate", found["estimate"])
creatinine = quiet(sdata.table, "../data/fetched/creatinine.csv", separator=",")
for name, table, a, b in [(ds, anxiety, "rater1", "rater2"), ("creatinine, serum and plasma (R mcr)", creatinine, "serum", "plasma")]:
    checks.check_values(name, 'agreement_between (test_method="lin")', answer(table.agreement_between, a, b, test_method="lin"))

agrees  anxiety (R irr example) | agreement_between (raters="a sample", checking_for="agreement", of="one rater") | estimate: simplydata 0.197998259356, McGraw & Wong (1996), NumPy 0.197998259356; relative difference 1.40e-16 (allowed 1e-10)
      Source: McGraw & Wong (1996), NumPy (run reference)
agrees  anxiety (R irr example) | agreement_between (raters="a sample", checking_for="agreement", of="one rater") | estimate: simplydata 0.197998259356, R irr icc (help page example) 0.197998259356; relative difference 1.40e-16 (allowed 1e-10)
      Source: R irr icc (help page example) (run reference) - https://search.r-project.org/CRAN/refmans/irr/html/icc.html (version 0.85)
agrees  anxiety (R irr example) | agreement_between (raters="a sample", checking_for="agreement", of="one rater") | estimate: simplydata 0.197998259356, R psych ICC 0.197998259356; relative difference 1.40e-16 (allowed 1e-10)
      Source: R psych ICC (run reference) - https://search.r-project.org/CRAN/refmans/psych/h

agrees  anxiety (R irr example) | agreement_between (raters="a sample", checking_for="agreement", of="average of raters") | estimate: simplydata 0.425498753117, McGraw & Wong (1996), NumPy 0.425498753117; relative difference 2.61e-16 (allowed 1e-10)
      Source: McGraw & Wong (1996), NumPy (run reference)
agrees  anxiety (R irr example) | agreement_between (raters="a sample", checking_for="agreement", of="average of raters") | estimate: simplydata 0.425498753117, R irr icc (help page example) 0.425498753117; relative difference 2.61e-16 (allowed 1e-10)
      Source: R irr icc (help page example) (run reference) - https://search.r-project.org/CRAN/refmans/irr/html/icc.html (version 0.85)
agrees  anxiety (R irr example) | agreement_between (raters="a sample", checking_for="agreement", of="average of raters") | estimate: simplydata 0.425498753117, R psych ICC 0.425498753117; relative difference 0.00e+00 (allowed 1e-10)
      Source: R psych ICC (run reference) - https://search.r-project.

agrees  anxiety (R irr example) | agreement_between (raters="different per row", checking_for="agreement", of="average of raters") | estimate: simplydata 0.388925729443, McGraw & Wong (1996), NumPy 0.388925729443; relative difference 4.28e-16 (allowed 1e-10)
      Source: McGraw & Wong (1996), NumPy (run reference)
agrees  anxiety (R irr example) | agreement_between (raters="different per row", checking_for="agreement", of="average of raters") | estimate: simplydata 0.388925729443, R irr icc (help page example) 0.388925729443; relative difference 1.43e-16 (allowed 1e-10)
      Source: R irr icc (help page example) (run reference) - https://search.r-project.org/CRAN/refmans/irr/html/icc.html (version 0.85)
agrees  anxiety (R irr example) | agreement_between (raters="different per row", checking_for="agreement", of="average of raters") | estimate: simplydata 0.388925729443, R psych ICC 0.388925729443; relative difference 1.43e-16 (allowed 1e-10)
      Source: R psych ICC (run reference) 

agrees  anxiety (R irr example) | agreement_between (checking_for="consistency", interval=99) | estimate 99 % CI low: simplydata -0.116868455975, McGraw & Wong (1996), NumPy -0.116868455975; relative difference 7.12e-16 (allowed 1e-10)
      Source: McGraw & Wong (1996), NumPy (run reference)
agrees  anxiety (R irr example) | agreement_between (checking_for="consistency", interval=99) | estimate 99 % CI high: simplydata 0.610556362301, McGraw & Wong (1996), NumPy 0.610556362301; relative difference 0.00e+00 (allowed 1e-10)
      Source: McGraw & Wong (1996), NumPy (run reference)
agrees  anxiety (R irr example) | agreement_between (test_method="krippendorff") | estimate: simplydata 0.170098607889, Krippendorff (2011) coincidence matrix, NumPy (interval) 0.170098607889; relative difference 4.90e-16 (allowed 1e-10)
      Source: Krippendorff (2011) coincidence matrix, NumPy (interval) (run reference)
explained difference – reference (C26)  anxiety (R irr example) | agreement_between (tes

agrees  anxiety (R irr example) | agreement_between (test_method="lin") | estimate: simplydata 0.29676511955, Lin (1989) written out, NumPy 0.29676511955; relative difference 0.00e+00 (allowed 1e-10)
      Source: Lin (1989) written out, NumPy (run reference)
agrees  anxiety (R irr example) | agreement_between (test_method="lin") | estimate 95 % CI low: simplydata -0.14982499739, Lin (1989) written out, NumPy -0.14982499739; relative difference 1.85e-16 (allowed 1e-10)
      Source: Lin (1989) written out, NumPy (run reference)
agrees  anxiety (R irr example) | agreement_between (test_method="lin") | estimate 95 % CI high: simplydata 0.642781085921, Lin (1989) written out, NumPy 0.642781085921; relative difference 0.00e+00 (allowed 1e-10)
      Source: Lin (1989) written out, NumPy (run reference)
agrees  creatinine, serum and plasma (R mcr) | agreement_between (test_method="lin") | estimate: simplydata 0.943985284537, Lin (1989) written out, NumPy 0.943985284537; relative difference 1

## Bland-Altman limits of agreement and Cronbach's alpha

In [5]:
for interval in (95, 90):
    checks.check_values("creatinine, serum and plasma (R mcr)", f"limits_of_agreement (interval={interval})",
                        answer(creatinine.limits_of_agreement, "serum", "plasma", interval=interval))
holzinger = quiet(sdata.table, "../data/fetched/holzinger_swineford_1939.csv", separator=",")
for name, table, items in [("Holzinger & Swineford (1939) x4, x5, x6", holzinger, ["x4", "x5", "x6"]),
                           (ds, anxiety, ["rater1", "rater2", "rater3"])]:
    for interval in (95, 90):
        found = read_rows(quiet(table.reliability_of, items, interval=interval))
        checks.check_values(name, f"reliability_of (interval={interval})", found[0])
        if interval == 95:
            checks.check_values(name, "reliability_of", {row["measure"]: row["estimate"] for row in found[1:]})

agrees  creatinine, serum and plasma (R mcr) | limits_of_agreement (interval=95) | average difference: simplydata -0.00768518518519, Bland & Altman (1986, 1999) written out, NumPy -0.00768518518519; relative difference 1.02e-14 (allowed 1e-10)
      Source: Bland & Altman (1986, 1999) written out, NumPy (run reference)
agrees  creatinine, serum and plasma (R mcr) | limits_of_agreement (interval=95) | average difference 95 % CI low: simplydata -0.0375226673394, Bland & Altman (1986, 1999) written out, NumPy -0.0375226673394; relative difference 1.85e-15 (allowed 1e-10)
      Source: Bland & Altman (1986, 1999) written out, NumPy (run reference)
agrees  creatinine, serum and plasma (R mcr) | limits_of_agreement (interval=95) | average difference 95 % CI high: simplydata 0.0221522969691, Bland & Altman (1986, 1999) written out, NumPy 0.0221522969691; relative difference 4.70e-16 (allowed 1e-10)
      Source: Bland & Altman (1986, 1999) written out, NumPy (run reference)
agrees  creatinine

## ROC analysis on pROC's `aSAH`, a chosen cut-off, and two scores (DeLong's paired test)

In [6]:
patients = load("asah")
patients.compare_to_truth("s100b", truth="outcome")

AUC 0.731 (95 % CI 0.630 to 0.833): a person who has it scores higher than one who does not in 73 of 100 such pairs (0.5 is no better than tossing a coin, 1 is perfect).
Best cut-off by Youden's index (sensitivity + specificity - 1): "s100b" 0.205 or more counts as positive (sensitivity 63.4 %, specificity 80.6 %). A cut-off chosen on these data looks better here than it will on new data.


Of 1000 people like these (36.3 % have it in this table), 363 have it: the test finds 230 (sensitivity 63.4 %, 95 % CI 46.9 to 77.9 %) and misses 133. Of the 637 without it, 513 test negative (specificity 80.6 %, 95 % CI 69.5 to 88.9 %) and 124 test positive by mistake.
So of the 354 positive results, 230 (65.0 %, 95 % CI 48.3 to 79.4 %) really have it; of the 646 negative results, 513 (79.5 %, 95 % CI 68.4 to 88.0 %) really do not.
A positive result is 3.26 times more common in people who have it than in people who do not (positive likelihood ratio, 95 % CI 1.93 to 5.51); a negative result is 0.454 times as common (negative likelihood ratio, 0.299 to 0.690).
The share of positives that are real depends on how common the condition is: give prevalence=1 to see it for a group where 1 % have it.
Method: exact Clopper-Pearson intervals (statsmodels proportion_confint), likelihood ratios from R epiR::epi.tests(method = "exact"); "s100b" 0.205 or more as positive against "outcome" = "Poor" (

,measure,estimate,estimate 95 % CI low,estimate 95 % CI high,what it means,method,count,out of
1,area under the ROC curve (AUC),0.731369,0.630118,0.832619,the chance that a person who has it scores higher,"R pROC::ci.auc(method = ""delong"")",NaN,NaN
2,best cut-off (Youden's index),0.205000,NaN,NaN,None,"R pROC::coords(""best"", best.method = ""youden"")",NaN,NaN
3,sensitivity,0.634146,0.469363,0.778772,"of those who have it, the share the test finds",exact Clopper-Pearson interval (statsmodels pr...,26.0,41.0
4,specificity,0.805556,0.695331,0.889416,"of those without it, the share that test negative",exact Clopper-Pearson interval (statsmodels pr...,58.0,72.0
5,positive predictive value,0.650000,0.483156,0.793718,"of the positive results, the share that really...",exact Clopper-Pearson interval (statsmodels pr...,26.0,40.0
6,negative predictive value,0.794521,0.683838,0.880187,"of the negative results, the share that really...",exact Clopper-Pearson interval (statsmodels pr...,58.0,73.0
7,positive likelihood ratio,3.261324,1.930236,5.510328,how many times more common a positive result i...,"R epiR::epi.tests(method = ""exact"")",NaN,NaN
8,negative likelihood ratio,0.454163,0.298805,0.690298,how many times as common a negative result is ...,"R epiR::epi.tests(method = ""exact"")",NaN,NaN
9,diagnostic odds ratio,7.180952,3.030133,17.017758,the odds of a positive result in those who hav...,"R epiR::epi.tests(method = ""exact"")",NaN,NaN


In [7]:
ds = "aSAH (R pROC example; Turck et al. 2010)"
for interval in (95, 90):
    checks.check_values(ds, f"compare_to_truth (a score, interval={interval})",
                        by_measure(patients.compare_to_truth, "s100b", truth="outcome", positive="Poor", interval=interval))
checks.check_values(ds, "compare_to_truth (a score, cut_off=0.3, prevalence=10)",
                    by_measure(patients.compare_to_truth, "s100b", truth="outcome", positive="Poor", cut_off=0.3, prevalence=10))
for interval, method in [(95, "compare_accuracy"), (90, "compare_accuracy (interval=90)")]:
    found = {row["measure"]: row for row in read_rows(quiet(patients.compare_accuracy, "s100b", "ndka", truth="outcome",
                                                            positive="Poor", interval=interval))}
    checks.check_values(ds, method, found["difference in AUC (s100b minus ndka)"])

agrees  aSAH (R pROC example; Turck et al. 2010) | compare_to_truth (a score, interval=95) | AUC: estimate: simplydata 0.731368563686, Mann-Whitney AUC and DeLong (1988) variance, NumPy 0.731368563686; relative difference 0.00e+00 (allowed 1e-10)
      Source: Mann-Whitney AUC and DeLong (1988) variance, NumPy (run reference)
agrees  aSAH (R pROC example; Turck et al. 2010) | compare_to_truth (a score, interval=95) | AUC: estimate: simplydata 0.731368563686, R pROC ci.auc (help page example) 0.731368563686; relative difference 0.00e+00 (allowed 1e-10)
      Source: R pROC ci.auc (help page example) (run reference) - https://search.r-project.org/CRAN/refmans/pROC/html/ci.auc.html (version 1.19.1)
agrees  aSAH (R pROC example; Turck et al. 2010) | compare_to_truth (a score, interval=95) | AUC: estimate 95 % CI low: simplydata 0.630118211762, Mann-Whitney AUC and DeLong (1988) variance, NumPy 0.630118211762; relative difference 1.76e-16 (allowed 1e-10)
      Source: Mann-Whitney AUC and D

## A yes/no test: epiR's `epi.tests` help page (Scott et al. 2008, Table 1)

As rows (`table.compare_to_truth`) and as the four counts (`sdata.compare_to_truth`; the counts read from the rows
with simplydata's `count_rows`), with the predictive values where 5 % have it; and from percents alone.

In [8]:
scott = load("scott_rows")
cells = {(row["test"], row["truth"]): row for row in read_rows(quiet(scott.count_rows, for_each_group_in="test", across="truth"))} \
    if False else None
counts = read_rows(quiet(scott.count_rows, for_each_group_in="test", across="truth"))
table = {row["test"]: row for row in counts}
found_, missed = table["yes"]["yes"], table["no"]["yes"]
false_alarms, correct_negatives = table["yes"]["no"], table["no"]["no"]
ds = "Scott et al. (2008), R epiR epi.tests example"
checks.check_values(ds, "table.compare_to_truth (a yes/no test column)",
                    by_measure(scott.compare_to_truth, "test", truth="truth", positive="yes"))
checks.check_values(ds, "sdata.compare_to_truth (four counts, prevalence=5)",
                    by_measure(sdata.compare_to_truth, found=found_, missed=missed, false_alarms=false_alarms,
                               correct_negatives=correct_negatives, prevalence=5))
ninety = by_measure(sdata.compare_to_truth, found=found_, missed=missed, false_alarms=false_alarms,
                    correct_negatives=correct_negatives, interval=90)
checks.check(ds, "sdata.compare_to_truth (interval=90)", "sensitivity 90 % CI low", ninety["sensitivity: estimate 90 % CI low"])
found = {row["measure"]: row["estimate"] for row in read_rows(quiet(sdata.compare_to_truth, sensitivity=90, specificity=95, prevalence=1))}
checks.check_values("sensitivity 90 %, specificity 95 %, prevalence 1 %",
                    "sdata.compare_to_truth (sensitivity=, specificity=, prevalence=)", found)

agrees  Scott et al. (2008), R epiR epi.tests example | table.compare_to_truth (a yes/no test column) | sensitivity: estimate: simplydata 0.900537634409, R binom.test 0.900537634409; relative difference 0.00e+00 (allowed 1e-10)
      Source: R binom.test (run reference) - https://search.r-project.org/R/refmans/stats/html/binom.test.html
agrees  Scott et al. (2008), R epiR epi.tests example | table.compare_to_truth (a yes/no test column) | sensitivity: estimate: simplydata 0.900537634409, R epiR epi.tests help page text 0.9; difference 5.38e-04 (allowed 5e-03)
      Source: R epiR epi.tests help page text (printed) - https://search.r-project.org/CRAN/refmans/epiR/html/epi.tests.html
agrees  Scott et al. (2008), R epiR epi.tests example | table.compare_to_truth (a yes/no test column) | sensitivity: estimate 95 % CI low: simplydata 0.876746176796, R binom.test 0.876746176796; relative difference 1.27e-16 (allowed 1e-10)
      Source: R binom.test (run reference) - https://search.r-project

2

## Result

In [9]:
checks.finish()

21_agreement_and_diagnostic_tests: 262 comparisons - 257 agrees, 5 explained difference – reference
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,21_agreement_and_diagnostic_tests,diagnoses (R irr example; Fleiss 1971),agreement_between (2 columns of categories: Co...,estimate,0.651163,0.651163,"Cohen (1960); Fleiss, Cohen & Everitt (1969), ...",0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,21_agreement_and_diagnostic_tests,diagnoses (R irr example; Fleiss 1971),agreement_between (2 columns of categories: Co...,estimate,0.651163,0.651163,R irr kappa2 (help page example),0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/irr/...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,21_agreement_and_diagnostic_tests,diagnoses (R irr example; Fleiss 1971),agreement_between (2 columns of categories: Co...,estimate 95 % CI low,0.455788,0.455788,"Cohen (1960); Fleiss, Cohen & Everitt (1969), ...",-5.551115e-17,1.000000e-10,relative,1.217915e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,21_agreement_and_diagnostic_tests,diagnoses (R irr example; Fleiss 1971),agreement_between (2 columns of categories: Co...,estimate 95 % CI high,0.846537,0.846537,"Cohen (1960); Fleiss, Cohen & Everitt (1969), ...",1.110223e-16,1.000000e-10,relative,1.311488e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,21_agreement_and_diagnostic_tests,diagnoses (R irr example; Fleiss 1971),agreement_between (2 columns of categories: Co...,statistic,6.996471,6.996471,"Cohen (1960); Fleiss, Cohen & Everitt (1969), ...",8.881784e-16,1.000000e-10,relative,1.269466e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,21_agreement_and_diagnostic_tests,"Scott et al. (2008), R epiR epi.tests example","sdata.compare_to_truth (four counts, prevalenc...",positive predictive value where 5 % have it,0.164972,0.164972,"Bayes' rule, NumPy",0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
258,21_agreement_and_diagnostic_tests,"Scott et al. (2008), R epiR epi.tests example","sdata.compare_to_truth (four counts, prevalenc...",negative predictive value where 5 % have it,0.993160,0.993160,"Bayes' rule, NumPy",1.110223e-16,1.000000e-10,relative,1.117869e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
259,21_agreement_and_diagnostic_tests,"Scott et al. (2008), R epiR epi.tests example",sdata.compare_to_truth (interval=90),sensitivity 90 % CI low,0.880617,0.880617,R binom.test,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
260,21_agreement_and_diagnostic_tests,"sensitivity 90 %, specificity 95 %, prevalence...","sdata.compare_to_truth (sensitivity=, specific...",positive predictive value,0.153846,0.153846,"Bayes' rule, NumPy",-5.551115e-17,1.000000e-10,relative,3.608225e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
